In [1]:
"""
SparsePCA 2D Grid Search: alpha x ridge_alpha
=============================================
Based on de Schipper & Van Deun (2021) model selection framework.

Table A: Model selection diagnostics
  - CV MSE (10-fold with one standard error rule)
  - BIC
  - Index of Sparseness
  - Non-zero weights
  - Which combination each method selects

Table B: Classification performance
  - F1-Score, Precision, Recall, Accuracy, Runtime
  - For the combination selected by each method

Pipeline: SparsePCA (n=32) → SVM classifier

"""

import time
import warnings
import numpy as np
import pandas as pd
import itertools

from sklearn.decomposition import SparsePCA
from sklearn.model_selection import train_test_split, KFold
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, mean_squared_error)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
np.random.seed(42)

# ─────────────────────────────────────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────
print("Loading data...")
X_raw = pd.read_excel('../minmax.xlsx').values.astype(np.float32)
y_raw = pd.read_csv('../idC_with_header.csv')['Label'].values

le = LabelEncoder()
y_raw = le.fit_transform(y_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42
)

print(f"Dataset: {X_raw.shape[0]} samples x {X_raw.shape[1]} features")
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}\n")

N_COMPONENTS = 32
N_FOLDS      = 10

# ─────────────────────────────────────────────────────────────────────────────
# 2. PARAMETER GRID
# ─────────────────────────────────────────────────────────────────────────────
# Alpha (λL): sparsity penalty — same range as previous tuning
# Ridge_alpha (λR): stability penalty — log-spaced as recommended by paper
alpha_values      = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
ridge_alpha_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

grid = list(itertools.product(alpha_values, ridge_alpha_values))
print(f"Grid: {len(alpha_values)} alpha x {len(ridge_alpha_values)} ridge_alpha = {len(grid)} combinations")
print(f"Each combination: {N_FOLDS}-fold CV\n")

# ─────────────────────────────────────────────────────────────────────────────
# 3. MODEL SELECTION HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def compute_bic(X, X_reconstructed, n_nonzero, n_samples):
    """
    BIC = RV_lambda/RV + df(lambda) * log(I)/I
    RV      = residual variance of full (non-sparse) PCA
    RV_lambda = residual variance of sparse PCA at given lambda
    df(lambda) = number of non-zero weights
    """
    RV_lambda = np.sum((X - X_reconstructed) ** 2)
    RV        = np.sum(X ** 2)  # baseline: predicting zero (max residual)
    bic       = (RV_lambda / RV) + n_nonzero * (np.log(n_samples) / n_samples)
    return bic

def compute_index_of_sparseness(X, X_reconstructed, X_pca_reconstructed,
                                 n_nonzero, n_components):
    """
    IS = VAF_pca * VAF_lambda * df(lambda) / (J * Q)
    VAF_pca    = variance accounted for by standard PCA
    VAF_lambda = variance accounted for by sparse PCA
    Higher IS = better balance of fit and sparsity
    """
    total_var      = np.sum(X ** 2)
    VAF_pca        = np.sum(X_pca_reconstructed ** 2) / total_var
    VAF_lambda     = np.sum(X_reconstructed ** 2) / total_var
    J              = X.shape[1]
    IS             = VAF_pca * VAF_lambda * (n_nonzero / (J * n_components))
    return IS

def get_pca_reconstruction(X, n_components):
    """Standard PCA reconstruction for IS baseline."""
    U, S, Vt = np.linalg.svd(X, full_matrices=False)
    U  = U[:, :n_components]
    S  = S[:n_components]
    Vt = Vt[:n_components, :]
    return (U * S) @ Vt

# ─────────────────────────────────────────────────────────────────────────────
# 4. MAIN GRID SEARCH
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 80)
print("RUNNING 2D GRID SEARCH")
print("=" * 80)

# PCA reconstruction baseline for IS computation
X_pca_recon = get_pca_reconstruction(X_train, N_COMPONENTS)

results = []
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

for idx, (alpha, ridge_alpha) in enumerate(grid):
    t0 = time.time()

    # ── 10-fold CV MSE ────────────────────────────────────────────────────
    fold_mse = []
    for train_idx, val_idx in kf.split(X_train):
        X_fold_train = X_train[train_idx]
        X_fold_val   = X_train[val_idx]

        try:
            spca_cv = SparsePCA(
                n_components=N_COMPONENTS,
                alpha=alpha,
                ridge_alpha=ridge_alpha,
                random_state=42,
                n_jobs=-1,
                max_iter=200
            )
            X_fold_latent = spca_cv.fit_transform(X_fold_train)
            # Reconstruct: X_recon = Z @ components
            X_recon_train = X_fold_latent @ spca_cv.components_
            # For val: transform then reconstruct
            X_val_latent  = spca_cv.transform(X_fold_val)
            X_recon_val   = X_val_latent @ spca_cv.components_
            mse = mean_squared_error(X_fold_val, X_recon_val)
            fold_mse.append(mse)
        except Exception:
            fold_mse.append(np.inf)

    cv_mse    = np.mean(fold_mse)
    cv_mse_se = np.std(fold_mse) / np.sqrt(N_FOLDS)

    # ── Full fit on training data ─────────────────────────────────────────
    try:
        spca = SparsePCA(
            n_components=N_COMPONENTS,
            alpha=alpha,
            ridge_alpha=ridge_alpha,
            random_state=42,
            n_jobs=-1,
            max_iter=200
        )
        X_train_latent = spca.fit_transform(X_train)
        X_test_latent  = spca.transform(X_test)

        # Reconstruction for BIC and IS
        X_recon        = X_train_latent @ spca.components_
        components_mat = spca.components_  # (n_components, n_features)
        n_nonzero      = int(np.sum(np.abs(components_mat) > 1e-10))

        bic = compute_bic(X_train, X_recon, n_nonzero, X_train.shape[0])
        IS  = compute_index_of_sparseness(
            X_train, X_recon, X_pca_recon, n_nonzero, N_COMPONENTS
        )

        # ── SVM classification ────────────────────────────────────────────
        clf = SVC(random_state=42)
        clf.fit(X_train_latent, y_train)
        y_pred = clf.predict(X_test_latent)

        acc  = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    except Exception as e:
        cv_mse = cv_mse_se = bic = IS = np.inf
        n_nonzero = 0
        acc = prec = rec = f1 = 0.0
        print(f"  ERROR at alpha={alpha}, ridge_alpha={ridge_alpha}: {e}")

    elapsed = time.time() - t0
    rt_str  = f"{elapsed:.0f} sec" if elapsed < 60 else f"{elapsed/60:.1f} min"

    results.append({
        "alpha"      : alpha,
        "ridge_alpha": ridge_alpha,
        "cv_mse"     : cv_mse,
        "cv_mse_se"  : cv_mse_se,
        "bic"        : bic,
        "IS"         : IS,
        "n_nonzero"  : n_nonzero,
        "accuracy"   : acc,
        "precision"  : prec,
        "recall"     : rec,
        "f1"         : f1,
        "runtime"    : rt_str,
    })

    print(f"[{idx+1:2d}/{len(grid)}] alpha={alpha:5.1f} ridge={ridge_alpha:6.3f} | "
          f"CV MSE={cv_mse:.4f} BIC={bic:.4f} IS={IS:.4f} "
          f"nonzero={n_nonzero:4d} Acc={acc*100:.2f}% ({rt_str})")

df = pd.DataFrame(results)

# ─────────────────────────────────────────────────────────────────────────────
# 5. MODEL SELECTION — apply each criterion
# ─────────────────────────────────────────────────────────────────────────────

# Filter out failed rows
df_valid = df[df["cv_mse"] < np.inf].copy()

# CV Best: lowest MSE
cv_best_idx   = df_valid["cv_mse"].idxmin()
cv_best_row   = df_valid.loc[cv_best_idx]

# CV 1SE rule: simplest model within 1 SE of best MSE
#   "simplest" = fewest non-zero weights still within best_MSE + 1*SE
cv_1se_thresh = cv_best_row["cv_mse"] + cv_best_row["cv_mse_se"]
candidates_1se = df_valid[df_valid["cv_mse"] <= cv_1se_thresh]
cv_1se_idx    = candidates_1se["n_nonzero"].idxmin()
cv_1se_row    = df_valid.loc[cv_1se_idx]

# BIC: lowest BIC
bic_best_idx  = df_valid["bic"].idxmin()
bic_best_row  = df_valid.loc[bic_best_idx]

# IS: highest IS
is_best_idx   = df_valid["IS"].idxmax()
is_best_row   = df_valid.loc[is_best_idx]

# Tag each row with which methods selected it
def tag_row(row_idx):
    tags = []
    if row_idx == cv_best_idx:  tags.append("CV Best")
    if row_idx == cv_1se_idx:   tags.append("CV 1SE ✓")
    if row_idx == bic_best_idx: tags.append("BIC")
    if row_idx == is_best_idx:  tags.append("IS")
    return ", ".join(tags) if tags else "—"

df_valid = df_valid.copy()
df_valid["selected_by"] = df_valid.index.map(tag_row)

# ─────────────────────────────────────────────────────────────────────────────
# 6. TABLE A — Model selection diagnostics
# ─────────────────────────────────────────────────────────────────────────────
print("\n\n" + "=" * 80)
print("TABLE A — Model Selection Diagnostics (de Schipper & Van Deun, 2021 framework)")
print("=" * 80)

df_tableA = df_valid[[
    "alpha", "ridge_alpha", "cv_mse", "bic", "IS", "n_nonzero", "selected_by"
]].copy()

df_tableA.columns = [
    "Alpha (λL)", "Ridge-alpha (λR)", "CV MSE (10-fold)",
    "BIC", "Index of Sparseness", "Non-zero weights", "Selected by"
]
df_tableA = df_tableA.round({"CV MSE (10-fold)": 4, "BIC": 4, "Index of Sparseness": 4})

print(df_tableA.to_string(index=False))
df_tableA.to_csv("SparsePCA_TableA_model_selection.csv", index=False)
print("\nSaved: SparsePCA_TableA_model_selection.csv")

# ─────────────────────────────────────────────────────────────────────────────
# 7. TABLE B — Classification performance of selected combinations
# ─────────────────────────────────────────────────────────────────────────────
print("\n\n" + "=" * 80)
print("TABLE B — Classification Performance of Selected Combinations (SVM)")
print("=" * 80)

selected_rows = []
for method, row in [("CV Best", cv_best_row),
                     ("CV 1SE rule", cv_1se_row),
                     ("BIC", bic_best_row),
                     ("Index of Sparseness", is_best_row)]:
    selected_rows.append({
        "Method"         : method,
        "Alpha (λL)"     : row["alpha"],
        "Ridge-alpha (λR)": row["ridge_alpha"],
        "F1-Score"       : round(row["f1"], 2),
        "Precision"      : round(row["precision"], 2),
        "Recall"         : round(row["recall"], 2),
        "Accuracy"       : f"{row['accuracy']*100:.2f}%",
        "Runtime"        : row["runtime"],
    })

df_tableB = pd.DataFrame(selected_rows)
print(df_tableB.to_string(index=False))
df_tableB.to_csv("SparsePCA_TableB_classification.csv", index=False)
print("\nSaved: SparsePCA_TableB_classification.csv")

# ─────────────────────────────────────────────────────────────────────────────
# 8. SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\n\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"\nCV Best      → alpha={cv_best_row['alpha']}, ridge_alpha={cv_best_row['ridge_alpha']}, "
      f"Acc={cv_best_row['accuracy']*100:.2f}%")
print(f"CV 1SE rule  → alpha={cv_1se_row['alpha']}, ridge_alpha={cv_1se_row['ridge_alpha']}, "
      f"Acc={cv_1se_row['accuracy']*100:.2f}%")
print(f"BIC          → alpha={bic_best_row['alpha']}, ridge_alpha={bic_best_row['ridge_alpha']}, "
      f"Acc={bic_best_row['accuracy']*100:.2f}%")
print(f"IS           → alpha={is_best_row['alpha']}, ridge_alpha={is_best_row['ridge_alpha']}, "
      f"Acc={is_best_row['accuracy']*100:.2f}%")

print(f"\nPaper recommendation: CV 1SE rule is preferred when component weights")
print(f"are to be interpreted. Best configuration under this rule:")
print(f"  alpha={cv_1se_row['alpha']}, ridge_alpha={cv_1se_row['ridge_alpha']}, "
      f"non-zero weights={int(cv_1se_row['n_nonzero'])}, "
      f"Accuracy={cv_1se_row['accuracy']*100:.2f}%")

# Save full results
df_valid.to_csv("SparsePCA_full_grid_results.csv", index=False)
print("\nFull grid results saved: SparsePCA_full_grid_results.csv")

Loading data...
Dataset: 443 samples x 900 features
Train: 354 | Test: 89

Grid: 6 alpha x 6 ridge_alpha = 36 combinations
Each combination: 10-fold CV

RUNNING 2D GRID SEARCH
[ 1/36] alpha=  0.1 ridge= 0.001 | CV MSE=0.0578 BIC=320.6248 IS=0.3289 nonzero=19311 Acc=92.13% (16.6 min)
[ 2/36] alpha=  0.1 ridge= 0.010 | CV MSE=0.0578 BIC=320.6248 IS=0.3243 nonzero=19311 Acc=92.13% (17.9 min)
[ 3/36] alpha=  0.1 ridge= 0.100 | CV MSE=0.0582 BIC=320.6282 IS=0.2841 nonzero=19311 Acc=92.13% (20.1 min)
[ 4/36] alpha=  0.1 ridge= 1.000 | CV MSE=0.0689 BIC=320.7275 IS=0.1165 nonzero=19311 Acc=94.38% (20.0 min)
[ 5/36] alpha=  0.1 ridge=10.000 | CV MSE=0.1031 BIC=321.0386 IS=0.0069 nonzero=19311 Acc=95.51% (17.3 min)
[ 6/36] alpha=  0.1 ridge=100.000 | CV MSE=0.1163 BIC=321.1580 IS=0.0001 nonzero=19311 Acc=95.51% (13.6 min)
[ 7/36] alpha=  0.5 ridge= 0.001 | CV MSE=0.0628 BIC=87.0538 IS=0.0797 nonzero=5220 Acc=93.26% (4.7 min)
[ 8/36] alpha=  0.5 ridge= 0.010 | CV MSE=0.0628 BIC=87.0539 IS=0.0786